In [3]:
!pip install tensorflow transformers torch nltk pandas numpy scikit-learn plotly google-generativeai

In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import nltk

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Bidirectional, LSTM, Dense, Dropout

print("Libraries imported successfully")

Libraries imported successfully


In [ ]:
!wget https://raw.githubusercontent.com/dair-ai/emotion_dataset/master/data/train.txt

--2026-07-03 07:47:16--  https://raw.githubusercontent.com/dair-ai/emotion_dataset/master/data/train.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 404 Not Found
2026-07-03 07:47:17 ERROR 404: Not Found.



In [7]:
import pandas as pd

url = "https://raw.githubusercontent.com/dair-ai/emotion_dataset/main/data/train.txt"

df = pd.read_csv(url, sep=';', names=['text', 'emotion'])

df.head()

HTTPError: HTTP Error 404: Not Found

In [ ]:
import pandas as pd

train_df = pd.read_csv('/content/train',
                       sep=';',
                       names=['text', 'emotion'])

train_df.head()

FileNotFoundError: [Errno 2] No such file or directory: '/content/train'

In [ ]:
!ls /content/

sample_data  train.txt


In [ ]:
import pandas as pd

train_df = pd.read_csv('/content/train.txt',
                       sep=';',
                       names=['text', 'emotion'])

train_df.head()

FileNotFoundError: [Errno 2] No such file or directory: '/content/train.txt'

In [ ]:
X = train_df['text']
y = train_df['emotion']

print(X.head())
print(y.head())

0                              i didnt feel humiliated
1    i can go from feeling so hopeless to so damned...
2     im grabbing a minute to post i feel greedy wrong
3    i am ever feeling nostalgic about the fireplac...
4                                 i am feeling grouchy
Name: text, dtype: object
0    sadness
1    sadness
2      anger
3       love
4      anger
Name: emotion, dtype: object


In [ ]:
from sklearn.preprocessing import LabelEncoder

encoder = LabelEncoder()
y = encoder.fit_transform(y)

print(y[:10])

[4 4 0 3 0 4 5 1 2 3]


In [ ]:
print(encoder.classes_)

['anger' 'fear' 'joy' 'love' 'sadness' 'surprise']


In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Create tokenizer
tokenizer = Tokenizer(num_words=10000)
tokenizer.fit_on_texts(X)

# Convert text to sequences
sequences = tokenizer.texts_to_sequences(X)

# Pad sequences
X = pad_sequences(sequences, maxlen=100)

print(X.shape)

(16000, 100)


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Training data shape:", X_train.shape)
print("Testing data shape:", X_test.shape)

Training data shape: (12800, 100)
Testing data shape: (3200, 100)


In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Bidirectional, LSTM, Dense, Dropout

model = Sequential([
    Embedding(input_dim=10000, output_dim=128, input_length=100),

    Bidirectional(LSTM(64)),

    Dropout(0.5),

    Dense(64, activation='relu'),

    Dense(6, activation='softmax')
])

model.compile(
    loss='sparse_categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [ ]:
history = model.fit(
    X_train,
    y_train,
    epochs=5,
    batch_size=32,
    validation_data=(X_test, y_test)
)

Epoch 1/5
400/400 ━━━━━━━━━━━━━━━━━━━━ 14s 16ms/step - accuracy: 0.5104 - loss: 1.2606 - val_accuracy: 0.7331 - val_loss: 0.7214
Epoch 2/5
400/400 ━━━━━━━━━━━━━━━━━━━━ 15s 15ms/step - accuracy: 0.8370 - loss: 0.4625 - val_accuracy: 0.8641 - val_loss: 0.3911
Epoch 3/5
400/400 ━━━━━━━━━━━━━━━━━━━━ 5s 12ms/step - accuracy: 0.9323 - loss: 0.2064 - val_accuracy: 0.8950 - val_loss: 0.3223
Epoch 4/5
400/400 ━━━━━━━━━━━━━━━━━━━━ 6s 15ms/step - accuracy: 0.9480 - loss: 0.1548 - val_accuracy: 0.9084 - val_loss: 0.3077
Epoch 5/5
400/400 ━━━━━━━━━━━━━━━━━━━━ 5s 13ms/step - accuracy: 0.9689 - loss: 0.0954 - val_accuracy: 0.9125 - val_loss: 0.3092


In [ ]:
model.save('bilstm_model.h5')

print("Model saved successfully!")

Model saved successfully!


In [ ]:
sample_text = ["I am very happy today"]

sequence = tokenizer.texts_to_sequences(sample_text)
padded = pad_sequences(sequence, maxlen=100)

prediction = model.predict(padded)

predicted_class = prediction.argmax(axis=1)[0]

emotion = encoder.inverse_transform([predicted_class])

print("Predicted Emotion:", emotion[0])

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 311ms/step
Predicted Emotion: joy


In [ ]:
import pickle

# Save tokenizer
with open('tokenizer.pkl', 'wb') as f:
    pickle.dump(tokenizer, f)

# Save label encoder
with open('label_encoder.pkl', 'wb') as f:
    pickle.dump(encoder, f)

print("Tokenizer and Label Encoder saved successfully!")

Tokenizer and Label Encoder saved successfully!


In [ ]:
# Predict on test data
y_pred = model.predict(X_test)

# Convert probabilities to class labels
y_pred_classes = np.argmax(y_pred, axis=1)

print("Predictions completed successfully!")

100/100 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step
Predictions completed successfully!


In [ ]:
import plotly.express as px

# Convert predicted labels back to emotion names
emotion_labels = encoder.inverse_transform(y_pred_classes)

# Create DataFrame
results_df = pd.DataFrame({
    'Emotion': emotion_labels
})

# Count emotions
emotion_counts = results_df['Emotion'].value_counts().reset_index()
emotion_counts.columns = ['Emotion', 'Count']

# Create bar chart
fig = px.bar(
    emotion_counts,
    x='Emotion',
    y='Count',
    title='Emotion Distribution',
    color='Emotion'
)

fig.show()

In [ ]:
from datetime import datetime

# Create logs dataframe
logs_df = pd.DataFrame({
    'Predicted_Emotion': emotion_labels
})

# Add timestamp
logs_df['Timestamp'] = datetime.now()

# Save logs to CSV
logs_df.to_csv('emotion_logs.csv', index=False)

print("Emotion logs saved successfully!")

Emotion logs saved successfully!


In [ ]:
def learning_support(emotion):

    if emotion == 'sadness':
        return "You seem sad. Take a short break and stay positive."

    elif emotion == 'anger':
        return "You seem frustrated. Try revising the topic slowly and relax."

    elif emotion == 'fear':
        return "Don't worry. Practice more and build confidence."

    elif emotion == 'joy':
        return "Great! You are feeling happy. Continue your learning journey."

    elif emotion == 'love':
        return "You are feeling positive. Keep learning with enthusiasm."

    elif emotion == 'surprise':
        return "You seem surprised. Explore more about this topic."

    else:
        return "Stay motivated and continue learning."

In [ ]:
emotion = "joy"

print("Emotion:", emotion)
print("AI Guidance:", learning_support(emotion))

Emotion: joy
AI Guidance: Great! You are feeling happy. Continue your learning journey.


In [ ]:
emotion = "sadness"

print("Emotion:", emotion)
print("AI Guidance:", learning_support(emotion))

Emotion: sadness
AI Guidance: You seem sad. Take a short break and stay positive.


In [ ]:
def learning_support(emotion):

    if emotion == 'sadness':
        return "You seem sad. Take a short break and stay positive."

    elif emotion == 'anger':
        return "You seem frustrated. Try revising the topic slowly and relax."

    elif emotion == 'fear':
        return "Don't worry. Practice more and build confidence."

    elif emotion == 'joy':
        return "Great! You are feeling happy. Continue your learning journey."

    elif emotion == 'love':
        return "You are feeling positive. Keep learning with enthusiasm."

    elif emotion == 'surprise':
        return "You seem surprised. Explore more about this topic."

    elif emotion == 'confused':
        return "You seem confused. Revise the concepts and watch tutorial videos."

    elif emotion == 'frustrated':
        return "Take a short break, relax, and try solving simpler problems first."

    elif emotion == 'curious':
        return "Great curiosity! Explore advanced topics and experiment with new ideas."

    elif emotion == 'confident':
        return "Excellent! You are confident. Move on to more challenging topics."

    elif emotion == 'bored':
        return "Try interactive learning methods such as videos, quizzes, or projects."

    else:
        return "Stay motivated and continue learning."

In [ ]:
emotion = "confused"

print("Emotion:", emotion)
print("AI Guidance:", learning_support(emotion))

Emotion: confused
AI Guidance: You seem confused. Revise the concepts and watch tutorial videos.


In [ ]:
# User input
user_text = input("Enter how you feel: ")

# Convert text to sequence
sequence = tokenizer.texts_to_sequences([user_text])

# Pad sequence
padded = pad_sequences(sequence, maxlen=100)

# Predict emotion
prediction = model.predict(padded)

predicted_class = np.argmax(prediction)

emotion = encoder.inverse_transform([predicted_class])[0]

# Display results
print("\nPredicted Emotion:", emotion)
print("AI Guidance:", learning_support(emotion))

Enter how you feel: i am happy


NameError: name 'tokenizer' is not defined

In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

tokenizer = Tokenizer(num_words=10000)
tokenizer.fit_on_texts(train_df['text'])

sequences = tokenizer.texts_to_sequences(train_df['text'])
X = pad_sequences(sequences, maxlen=100)

print("Tokenizer recreated successfully!")from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

tokenizer = Tokenizer(num_words=10000)
tokenizer.fit_on_texts(train_df['text'])

sequences = tokenizer.texts_to_sequences(train_df['text'])
X = pad_sequences(sequences, maxlen=100)

print("Tokenizer recreated successfully!")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

tokenizer = Tokenizer(num_words=10000)
tokenizer.fit_on_texts(train_df['text'])

sequences = tokenizer.texts_to_sequences(train_df['text'])
X = pad_sequences(sequences, maxlen=100)

print("Tokenizer recreated successfully!")

NameError: name 'train_df' is not defined

In [ ]:
print(type(model))

NameError: name 'model' is not defined

In [ ]:
print(type(model))

NameError: name 'model' is not defined

In [ ]:
import pickle
from tensorflow.keras.models import load_model

# Load model
model = load_model('/content/bilstm_model.h5')

# Load tokenizer
with open('/content/tokenizer.pkl', 'rb') as f:
    tokenizer = pickle.load(f)

# Load label encoder
with open('/content/label_encoder.pkl', 'rb') as f:
    encoder = pickle.load(f)

print("Model, Tokenizer, and Encoder loaded successfully!")

FileNotFoundError: [Errno 2] Unable to synchronously open file (unable to open file: name = '/content/bilstm_model.h5', errno = 2, error message = 'No such file or directory', flags = 0, o_flags = 0)

In [ ]:
!ls /content/

sample_data


In [ ]:
import pandas as pd

train_df = pd.read_csv(
    '/content/train.txt',
    sep=';',
    names=['text', 'emotion']
)

train_df.head()

FileNotFoundError: [Errno 2] No such file or directory: '/content/train.txt'

In [8]:
!ls /content/

sample_data  train.txt


In [10]:
import pandas as pd

train_df = pd.read_csv(
    '/content/train.txt',
    sep=';',
    names=['text', 'emotion']
)

train_df.head()

,text,emotion
0,i didnt feel humiliated,sadness
1,i can go from feeling so hopeless to so damned...,sadness
2,im grabbing a minute to post i feel greedy wrong,anger
3,i am ever feeling nostalgic about the fireplac...,love
4,i am feeling grouchy,anger


In [11]:
X = train_df['text']
y = train_df['emotion']

print(X.head())
print(y.head())

0                              i didnt feel humiliated
1    i can go from feeling so hopeless to so damned...
2     im grabbing a minute to post i feel greedy wrong
3    i am ever feeling nostalgic about the fireplac...
4                                 i am feeling grouchy
Name: text, dtype: object
0    sadness
1    sadness
2      anger
3       love
4      anger
Name: emotion, dtype: object


In [12]:
from sklearn.preprocessing import LabelEncoder

encoder = LabelEncoder()
y = encoder.fit_transform(y)

print(y[:10])

[4 4 0 3 0 4 5 1 2 3]


In [13]:
print(encoder.classes_)

['anger' 'fear' 'joy' 'love' 'sadness' 'surprise']


In [14]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Create tokenizer
tokenizer = Tokenizer(num_words=10000)
tokenizer.fit_on_texts(X)

# Convert text to sequences
sequences = tokenizer.texts_to_sequences(X)

# Pad sequences
X = pad_sequences(sequences, maxlen=100)

print(X.shape)

(16000, 100)


In [15]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Training data shape:", X_train.shape)
print("Testing data shape:", X_test.shape)

Training data shape: (12800, 100)
Testing data shape: (3200, 100)


In [16]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Bidirectional, LSTM, Dense, Dropout

model = Sequential([
    Embedding(input_dim=10000, output_dim=128),

    Bidirectional(LSTM(64)),

    Dropout(0.5),

    Dense(64, activation='relu'),

    Dense(6, activation='softmax')
])

model.compile(
    loss='sparse_categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [17]:
history = model.fit(
    X_train,
    y_train,
    epochs=5,
    batch_size=32,
    validation_data=(X_test, y_test)
)

Epoch 1/5
400/400 ━━━━━━━━━━━━━━━━━━━━ 14s 15ms/step - accuracy: 0.5225 - loss: 1.2244 - val_accuracy: 0.7169 - val_loss: 0.7326
Epoch 2/5
400/400 ━━━━━━━━━━━━━━━━━━━━ 6s 16ms/step - accuracy: 0.8560 - loss: 0.4164 - val_accuracy: 0.8856 - val_loss: 0.3397
Epoch 3/5
400/400 ━━━━━━━━━━━━━━━━━━━━ 5s 14ms/step - accuracy: 0.9350 - loss: 0.1957 - val_accuracy: 0.9056 - val_loss: 0.2820
Epoch 4/5
400/400 ━━━━━━━━━━━━━━━━━━━━ 7s 16ms/step - accuracy: 0.9615 - loss: 0.1166 - val_accuracy: 0.9062 - val_loss: 0.3491
Epoch 5/5
400/400 ━━━━━━━━━━━━━━━━━━━━ 5s 14ms/step - accuracy: 0.9687 - loss: 0.0985 - val_accuracy: 0.8706 - val_loss: 0.4553


In [18]:
model.save('bilstm_model.h5')

print("Model saved successfully!")

Model saved successfully!


In [19]:
import pickle

# Save tokenizer
with open('tokenizer.pkl', 'wb') as f:
    pickle.dump(tokenizer, f)

# Save label encoder
with open('label_encoder.pkl', 'wb') as f:
    pickle.dump(encoder, f)

print("Tokenizer and Label Encoder saved successfully!")

Tokenizer and Label Encoder saved successfully!


In [20]:
sample_text = ["I am very happy because I completed my project"]

sequence = tokenizer.texts_to_sequences(sample_text)
padded = pad_sequences(sequence, maxlen=100)

prediction = model.predict(padded)

predicted_class = prediction.argmax(axis=1)[0]

emotion = encoder.inverse_transform([predicted_class])

print("Predicted Emotion:", emotion[0])

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 585ms/step
Predicted Emotion: joy


In [21]:
# Predict on test data
y_pred = model.predict(X_test)

# Convert probabilities to class labels
y_pred_classes = np.argmax(y_pred, axis=1)

print("Predictions completed successfully!")

100/100 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step
Predictions completed successfully!


In [22]:
import plotly.express as px
import pandas as pd

# Convert predicted labels back to emotion names
emotion_labels = encoder.inverse_transform(y_pred_classes)

# Create DataFrame
results_df = pd.DataFrame({
    'Emotion': emotion_labels
})

# Count emotions
emotion_counts = results_df['Emotion'].value_counts().reset_index()
emotion_counts.columns = ['Emotion', 'Count']

# Create chart
fig = px.bar(
    emotion_counts,
    x='Emotion',
    y='Count',
    title='Emotion Distribution',
    color='Emotion'
)

fig.show()

In [23]:
from datetime import datetime

# Create logs DataFrame
logs_df = pd.DataFrame({
    'Predicted_Emotion': emotion_labels
})

# Add timestamp
logs_df['Timestamp'] = datetime.now()

# Save to CSV
logs_df.to_csv('emotion_logs.csv', index=False)

print("Emotion logs saved successfully!")

Emotion logs saved successfully!


In [24]:
def learning_support(emotion):

    if emotion == 'sadness':
        return "You seem sad. Take a short break and stay positive."

    elif emotion == 'anger':
        return "You seem frustrated. Try revising the topic slowly and relax."

    elif emotion == 'fear':
        return "Don't worry. Practice more and build confidence."

    elif emotion == 'joy':
        return "Great! You are feeling happy. Continue your learning journey."

    elif emotion == 'love':
        return "You are feeling positive. Keep learning with enthusiasm."

    elif emotion == 'surprise':
        return "You seem surprised. Explore more about this topic."

    elif emotion == 'confused':
        return "You seem confused. Revise the concepts and watch tutorial videos."

    elif emotion == 'frustrated':
        return "Take a short break, relax, and try solving simpler problems first."

    elif emotion == 'curious':
        return "Great curiosity! Explore advanced topics and experiment with new ideas."

    elif emotion == 'confident':
        return "Excellent! You are confident. Move on to more challenging topics."

    elif emotion == 'bored':
        return "Try interactive learning methods such as videos, quizzes, or projects."

    else:
        return "Stay motivated and continue learning."

In [25]:
emotion = "joy"

print("Emotion:", emotion)
print("AI Guidance:", learning_support(emotion))

Emotion: joy
AI Guidance: Great! You are feeling happy. Continue your learning journey.


In [26]:
user_text = input("Enter how you feel: ")

# Convert text to sequence
sequence = tokenizer.texts_to_sequences([user_text])

# Pad sequence
padded = pad_sequences(sequence, maxlen=100)

# Predict emotion
prediction = model.predict(padded)

predicted_class = np.argmax(prediction)

emotion = encoder.inverse_transform([predicted_class])[0]

# Display results
print("\nPredicted Emotion:", emotion)
print("AI Guidance:", learning_support(emotion))

Enter how you feel: I am very happy because I completed my project
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step

Predicted Emotion: joy
AI Guidance: Great! You are feeling happy. Continue your learning journey.
